In [1]:
import json
import re
import time
import requests
import pandas as pd
from bs4 import BeautifulSoup

In [2]:
def scraper_transfermarkt_plantel(url, nombre_pais):
    # User-Agent obligatorio para evitar bloqueos inmediatos
    headers = {
        "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36"
    }

    try:
        response = requests.get(url, headers=headers)
        response.raise_for_status()
    except requests.exceptions.RequestException as e:
        print(f"  -> Error al conectar con {nombre_pais}: {e}")
        return None

    soup = BeautifulSoup(response.text, "html.parser")

    # Buscamos la tabla principal de jugadores
    tabla = soup.find("table", class_="items")
    if not tabla:
        print(f"  -> No se encontró la tabla de jugadores para {nombre_pais}.")
        return None

    tbody = tabla.find("tbody")
    datos_jugadores = []

    # Iteramos sobre las filas principales de la tabla
    for fila in tbody.find_all("tr", recursive=False):
        columnas = fila.find_all("td", recursive=False)

        if len(columnas) < 5:
            continue

        # 1. Número / Dorsal
        numero = columnas[0].get_text(strip=True)

        # 2. Jugador y Posición
        celda_jugador = columnas[1]
        td_nombre = celda_jugador.find("td", class_="hauptlink")

        if not td_nombre:
            continue

        nombre = td_nombre.get_text(strip=True)
        posicion = celda_jugador.find_all("tr")[1].get_text(strip=True)

        # 3. Edad
        edad = columnas[2].get_text(strip=True)

        # 4. Club
        img_club = columnas[3].find("img")
        club = (
            img_club["title"]
            if img_club and "title" in img_club.attrs
            else "Libre/Sin Club"
        )

        # 5. Valor de Mercado
        valor_mercado = columnas[4].get_text(strip=True)

        datos_jugadores.append(
            {
                "#": numero,
                "Player": nombre,
                "Position": posicion,
                "Age": edad,
                "Club": club,
                "Market Value": valor_mercado,
            }
        )

    # Convertimos a DataFrame para mantener consistencia de datos
    df = pd.DataFrame(datos_jugadores)
    return df

In [3]:
paises = [
# Pais, Codigo Transfermarkt, Código URL
    ('France', 3377,'frankreich'),
    ('England', 3299,'england'),
    ('Spain', 3375,'spanien'),
    ('Germany', 3262,'deutschland'),
    ('Portugal', 3300,'portugal'),
    ('Brazil', 3439,'brasilien'),
    ('Netherlands', 3379,'niederlande'),
    ('Argentina', 3437,'argentinien'),
    ('Norway', 3440,'norwegen'),
    ('Belgium', 3382,'belgien'),
    ('Turkey', 3381,'turkei'),
    ('Côte d\'Ivoire',3591,'elfenbeinkuste'),
    ('Senegal', 3499,'senegal'),
    ('Sweden', 3557,'schweden'),
    ('Uruguay', 3449,'uruguay'),
    ('USA', 3505,'vereinigte-staaten'),
    ('Croatia', 3556,'kroatien'),
    ('Switzerland', 3384,'schweiz'),
    ('Colombia', 3816,'kolumbien'),
    ('Ghana', 3441,'ghana'),
    ('Japan', 3435,'japan'),
    ('Austria', 3383,'osterreich'),
    ('Ecuador', 5750,'ecuador'),
    ('Morocco', 3575,'marokko'),
    ('Algeria', 3614,'algerien'),
    ('Canada', 3510,'kanada'),
    ('Scotland', 3380,'schottland'),
    ('Czechia', 3445,'tschechien'),
    ('Congo DR', 3854,'demokratische-republik-kongo'),
    ('Korea Republic', 3589,'sudkorea'),
    ('Bosnia-Herzegovina', 3446,'bosnien-herzegowina'),
    ('Paraguay', 3581,'paraguay'),
    ('Egypt', 3672,'agypten'),
    ('Mexico', 6303,'mexiko'),
    ('Uzbekistan', 3563,'usbekistan'),
    ('Tunisia', 3670,'tunesien'),
    ('Cabo Verde', 4311,'kap-verde'),
    ('Haiti', 14161,'haiti'),
    ('South Africa', 3806,'sudafrika'),
    ('Australia', 3433,'australien'),
    ('IR Iran', 3582,'iran'),
    ('New Zealand', 9171,'neuseeland'),
    ('Panama', 3577,'panama'),
    ('Curaçao', 32364,'curacao'),
    ('Saudi Arabia', 3807,'saudi-arabien'),
    ('Qatar', 14162,'katar'),
    ('Iraq', 3560,'irak'),
    ('Jordan', 15737,'jordanien')
]

In [4]:
# Diccionario final donde guardaremos los resultados de todos los países
diccionario_final = {}
diccionario_df = {}

print(f"Iniciando el scraping de {len(paises)} países...\n")

for nombre, id_pais, cod in paises:
    # Construcción dinámica de la URL usando los datos de la lista
    # (Pasamos el nombre a minúsculas para que cumpla con el estándar de la URL)
    url_dinamica = f"https://www.transfermarkt.com/{cod.lower()}/startseite/verein/{id_pais}"

    # Llamamos al scraper para obtener el DataFrame del país actual
    df_pais = scraper_transfermarkt_plantel(url_dinamica, nombre)

    if df_pais is not None and not df_pais.empty:
        # Si el DataFrame es válido, lo convertimos a registros dicc/JSON
        # y lo asignamos como value de la key con el nombre del país
        diccionario_df[nombre] = df_pais
        diccionario_final[nombre] = df_pais.to_dict(orient="records")
        print(f"Done | {nombre}")
    else:
        print(f"  -> Saltando {nombre} debido a un error de lectura.")

    # PAUSA ANTIBAN: Esperamos 2 segundos antes de la siguiente petición
    # Esto evita que Transfermarkt identifique el comportamiento como un ataque DDoS
    time.sleep(2)

Iniciando el scraping de 48 países...

Done | France
Done | England
Done | Spain
Done | Germany
Done | Portugal
Done | Brazil
Done | Netherlands
Done | Argentina
Done | Norway
Done | Belgium
Done | Turkey
Done | Côte d'Ivoire
Done | Senegal
Done | Sweden
Done | Uruguay
Done | USA
Done | Croatia
Done | Switzerland
Done | Colombia
Done | Ghana
Done | Japan
Done | Austria
Done | Ecuador
Done | Morocco
Done | Algeria
Done | Canada
Done | Scotland
Done | Czechia
Done | Congo DR
Done | Korea Republic
Done | Bosnia-Herzegovina
Done | Paraguay
Done | Egypt
Done | Mexico
Done | Uzbekistan
Done | Tunisia
Done | Cabo Verde
Done | Haiti
Done | South Africa
Done | Australia
Done | IR Iran
Done | New Zealand
Done | Panama
Done | Curaçao
Done | Saudi Arabia
Done | Qatar
Done | Iraq
Done | Jordan


In [5]:
# ==========================================
# GUARDAR EL OUTPUT FINAL EN UN JSON
# ==========================================
if diccionario_final:
    archivo_salida = "../data/jugadores_todos_los_paises.json"
    with open(archivo_salida, "w", encoding="utf-8") as f:
        json.dump(diccionario_final, f, ensure_ascii=False, indent=4)

    print(
        f"\n=== PROCESO TERMINADO ==="
        f"\nSe ha generado el archivo unificado '{archivo_salida}' con éxito."
    )


=== PROCESO TERMINADO ===
Se ha generado el archivo unificado '../data/jugadores_todos_los_paises.json' con éxito.


In [6]:
diccionario_df['Iraq']

,#,Player,Position,Age,Club,Market Value
0,22,Ahmed Basil Fadhil,Goalkeeper,19/08/1996 (29),Al-Shorta SC,€450k
1,1,Fahad Talib,Goalkeeper,21/10/1994 (31),Al-Talaba SC,€400k
2,12,Kumel Al-Rekabe,Goalkeeper,19/08/2004 (21),Erbil SC,€250k
3,4,Zaid Tahseen,Centre-Back,29/01/2001 (25),Pakhtakor Tashkent,€600k
4,5,Akam Hashem,Centre-Back,16/08/1998 (27),Al-Zawraa SC,€600k
5,-,Maytham Jabbar,Centre-Back,10/11/2000 (25),Al-Zawraa SC,€400k
6,-,Manaf Younis,Centre-Back,16/11/1996 (29),Al-Shorta SC,€350k
7,6,Frans Dhia Putros,Centre-Back,14/07/1993 (32),PERSIB Bandung,€350k
8,2,Rebin Sulaka,Centre-Back,12/04/1992 (34),Port FC,€100k
9,23,Merchas Doski,Left-Back,07/12/1999 (26),FC Viktoria Plzen,€1.70m
